# Task 3: Correlation between News Sentiment and Stock Movement

## Objective
Quantify the relationship between financial news sentiment and daily stock price returns using statistical methods.

## Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from textblob import TextBlob
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Download VADER lexicon if not present
nltk.download('vader_lexicon', quiet=True)

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

## Load and Prepare Data

In [ ]:
# Load news data
news_df = pd.read_csv('../newsData/raw_analyst_ratings.csv', parse_dates=['date'])

# Filter for AAPL news
news_df = news_df[news_df['stock'] == 'AAPL'].copy()

# Load stock data
stock_df = pd.read_csv('../newsData/AAPL.csv', parse_dates=['Date'], index_col='Date')

print("News data shape:", news_df.shape)
print("Stock data shape:", stock_df.shape)
print("\nNews columns:", news_df.columns.tolist())
print("Stock columns:", stock_df.columns.tolist())

## Date Alignment

In [ ]:
# Function to align dates to next trading day
def align_to_trading_day(date, trading_dates):
    if date in trading_dates:
        return date
    # Find next trading day
    next_dates = trading_dates[trading_dates > date]
    if len(next_dates) > 0:
        return next_dates.min()
    return None

# Get trading dates from stock data
trading_dates = set(stock_df.index.date)

# Align news dates
news_df['aligned_date'] = news_df['date'].dt.date.apply(lambda x: align_to_trading_day(x, trading_dates))

# Drop rows where no trading day found
news_df = news_df.dropna(subset=['aligned_date'])

# Convert back to datetime
news_df['aligned_date'] = pd.to_datetime(news_df['aligned_date'])

print("Date alignment completed.")
print("News data after alignment:", news_df.shape)

## Sentiment Analysis

In [ ]:
# Initialize VADER sentiment analyzer
sid = SentimentIntensityAnalyzer()

# Function to get compound sentiment score
def get_sentiment_score(text):
    return sid.polarity_scores(text)['compound']

# Apply sentiment analysis
news_df['sentiment_score'] = news_df['headline'].apply(get_sentiment_score)

print("Sentiment analysis completed.")
print("Sentiment score range:", news_df['sentiment_score'].min(), "to", news_df['sentiment_score'].max())
print("Sample sentiments:")
print(news_df[['headline', 'sentiment_score']].head())

## Calculate Daily Stock Returns

In [ ]:
# Calculate daily returns
stock_df['daily_return'] = stock_df['Close'].pct_change() * 100

print("Daily returns calculated.")
print("Returns summary:")
print(stock_df['daily_return'].describe())

## Aggregate and Correlate

In [ ]:
# Aggregate sentiment by date
daily_sentiment = news_df.groupby('aligned_date')['sentiment_score'].mean().reset_index()
daily_sentiment.columns = ['date', 'avg_sentiment']

# Merge with stock returns
analysis_df = pd.merge(daily_sentiment, stock_df[['daily_return']], left_on='date', right_index=True, how='inner')

# Calculate correlation
correlation = analysis_df['avg_sentiment'].corr(analysis_df['daily_return'])

print("Correlation analysis:")
print(f"Pearson correlation coefficient: {correlation:.4f}")
print(f"Number of days with both sentiment and returns: {len(analysis_df)}")

# Classify sentiment
def classify_sentiment(score):
    if score > 0.05:
        return 'positive'
    elif score < -0.05:
        return 'negative'
    else:
        return 'neutral'

analysis_df['sentiment_category'] = analysis_df['avg_sentiment'].apply(classify_sentiment)

# Average returns by category
category_returns = analysis_df.groupby('sentiment_category')['daily_return'].mean()

print("\nAverage daily returns by sentiment category:")
print(category_returns)

## Visualize the Relationship

In [ ]:
# Scatter plot
plt.figure(figsize=(10, 6))
plt.scatter(analysis_df['avg_sentiment'], analysis_df['daily_return'], alpha=0.6)
plt.title(f'Sentiment vs Daily Returns (Correlation: {correlation:.4f})')
plt.xlabel('Average Daily Sentiment Score')
plt.ylabel('Daily Return (%)')
plt.grid(True, alpha=0.3)
plt.show()

# Bar chart of average returns by category
plt.figure(figsize=(8, 5))
category_returns.plot(kind='bar', color=['red', 'gray', 'green'])
plt.title('Average Daily Returns by Sentiment Category')
plt.xlabel('Sentiment Category')
plt.ylabel('Average Daily Return (%)')
plt.xticks(rotation=0)
plt.grid(True, alpha=0.3)
plt.show()

## Interpret Results

### Correlation Analysis Results

The Pearson correlation coefficient between average daily sentiment scores and daily stock returns is **[correlation_value]**. This indicates **[weak/moderate/strong] **[positive/negative]** correlation between news sentiment and stock price movements.

### Interpretation
- **Strength and Direction**: The correlation suggests that **[describe]**. A positive correlation would mean that positive news sentiment tends to coincide with positive stock returns, while negative sentiment correlates with declines.
- **Sentiment Categories**: Days with positive sentiment showed average returns of **[value]**, neutral days **[value]**, and negative days **[value]**. This pattern **[supports/contradicts]** the correlation finding.

### Limitations
- **Lag Effects**: News sentiment may influence stock prices with a delay, not captured in same-day analysis.
- **Confounding Factors**: Stock movements are influenced by many factors beyond news sentiment, including macroeconomic conditions, company fundamentals, and market trends.
- **Sentiment Analysis Accuracy**: VADER provides reasonable sentiment scores but may not capture nuanced financial context or sarcasm.
- **Data Coverage**: Analysis is limited to days with both news articles and trading data, potentially biasing results.

This analysis provides initial insights into the relationship between news sentiment and stock movements, but further research with more sophisticated models and additional data sources would be beneficial.